## 在 Database 建立所有 Tables

In [ ]:
CREATE TABLE GAME (
    game_id CHAR(3) PRIMARY KEY, -- '001'
    player_count INT,
    winner_side VARCHAR(20),
    assassin_target_seat_no INT,
    game_date DATE
);

CREATE TABLE PLAYER (
    game_id CHAR(3) REFERENCES GAME(game_id),
    seat_number INT,
    role VARCHAR(50),
    PRIMARY KEY (game_id, seat_number)
);

CREATE TABLE MISSION (
    game_id CHAR(3) REFERENCES GAME(game_id),
    mission_no INT,
    team_size INT,
    fail_count INT DEFAULT 0,       --預設0，流局再加一
    mission_result VARCHAR(20),
    PRIMARY KEY (game_id, mission_no)
);

CREATE TABLE PROPOSAL (
    game_id CHAR(3),
    mission_no INT,
    proposal_no INT,
    leader_seat_no INT,
    vote_result VARCHAR(20),
    yes_amount INT,
    no_amount INT,
    PRIMARY KEY (game_id, mission_no, proposal_no),
    FOREIGN KEY (game_id, mission_no) REFERENCES MISSION(game_id, mission_no)
);

CREATE TABLE PROPOSAL_CONTAINS (
    game_id CHAR(3),
    mission_no INT,
    proposal_no INT,
    seat_number INT,
    contains_or_not BOOLEAN DEFAULT FALSE,
    PRIMARY KEY (game_id, mission_no, proposal_no, seat_number),
    FOREIGN KEY (game_id, mission_no, proposal_no) REFERENCES PROPOSAL(game_id, mission_no, proposal_no),
    FOREIGN KEY (game_id, seat_number) REFERENCES PLAYER(game_id, seat_number)
);

CREATE TABLE VOTE_PROPOSAL (
    game_id CHAR(3),
    seat_number INT,
    mission_no INT,
    proposal_no INT,
    vote_y_n BOOLEAN,
    PRIMARY KEY (game_id, seat_number, mission_no, proposal_no),
    FOREIGN KEY (game_id, mission_no, proposal_no) REFERENCES PROPOSAL(game_id, mission_no, proposal_no),
    FOREIGN KEY (game_id, seat_number) REFERENCES PLAYER(game_id, seat_number)
);

CREATE TABLE BECOMES_MISSION (
    game_id CHAR(3),
    mission_no INT,
    proposal_no INT,
    succeed_amount INT,
    fail_amount INT,
    five_fails BOOLEAN DEFAULT FALSE,
    PRIMARY KEY (game_id, mission_no, proposal_no),
    FOREIGN KEY (game_id, mission_no) REFERENCES MISSION(game_id, mission_no),
    FOREIGN KEY (game_id, mission_no, proposal_no) REFERENCES PROPOSAL(game_id, mission_no, proposal_no)
);

### 建立遊戲
    INSERT INTO : GAME

In [ ]:
INSERT INTO GAME (game_id, player_count, game_date)
VALUES (
    '001',      -- 三位數Game ID
    5,          -- 5 ~ 10
    '2026-04-29'-- 遊戲進行日期
);

### 固定座位
    INSERT INTO : PLAYER

In [ ]:
INSERT INTO PLAYER (game_id, seat_number)
VALUES 
    ('001', 1),
    ('001', 2),
    ('001', 3),
    ('001', 4),
    ('001', 5),
    ('001', 6),
    ('001', 7),
    ('001', 8);

### 新回合
    INSERT INTO : ROUND

In [ ]:
INSERT INTO MISSION (game_id, mission_no, team_size)
VALUES (
    '001',      -- 三位數Game ID
    1,          -- 第幾Mission
    2           -- 2 ~ 5
);

### 發任務牌
    INSERT INTO : PROPOSAL

In [ ]:
INSERT INTO PROPOSAL (game_id, mission_no, proposal_no, leader_seat_no)
VALUES (
    '001',      -- 三位數 Game ID
    1,          -- 第幾Mission
    1,          -- 第幾Proposal
    1           -- 隊長座位
);

### 隊伍有誰
    INSERT INTO : PROPOSAL_CONTAINS

In [ ]:
INSERT INTO PROPOSAL_CONTAINS (game_id, mission_no, proposal_no, seat_number, contains_or_not)
VALUES 
    ('001', 1, 1, 1, TRUE),
    ('001', 1, 1, 2, TRUE),
    ('001', 1, 1, 3, TRUE),
    ('001', 1, 1, 4, TRUE),
    ('001', 1, 1, 5, TRUE),
    ('001', 1, 1, 6, TRUE),
    ('001', 1, 1, 7, TRUE),
    ('001', 1, 1, 8, TRUE)
    ;

### 表決
    INSERT INTO : VOTE_PROPOSAL

In [ ]:
INSERT INTO VOTE_PROPOSAL (game_id, mission_no, proposal_no, seat_number, vote_y_n)
VALUES
    ('001', 1, 1, 1, TRUE),
    ('001', 1, 1, 2, TRUE),
    ('001', 1, 1, 3, TRUE),
    ('001', 1, 1, 4, TRUE),
    ('001', 1, 1, 5, TRUE),
    ('001', 1, 1, 6, TRUE),
    ('001', 1, 1, 7, TRUE),
    ('001', 1, 1, 8, TRUE)
    ;

### 表決結果
    UPDATE : PROPOSAL

In [ ]:
UPDATE PROPOSAL
SET vote_result = 'Rejected',  -- Passed or Rejected
    yes_amount = 8,
    no_amount = 0
WHERE game_id = '001'
AND mission_no = 1
AND proposal_no = 1
;

### 流局

In [ ]:
UPDATE MISSION 
SET fail_count = fail_count + 1 
WHERE game_id = '001'
AND mission_no = 1
;

### 出任務
    INSERT INTO : BECOMES_MISSION

In [ ]:
INSERT INTO BECOMES_MISSION (game_id, mission_no, proposal_no, succeed_amount, fail_amount, five_fails)
VALUES (
    '001',
    1,
    1,
    2,
    0,
    FALSE   -- 是否因為五次失敗強制進入
);

### 任務結果
    UPDATE : MISSION

In [ ]:
UPDATE MISSION
SET mission_result = 'Success'  -- Success or Fail
WHERE game_id = '001'
AND mission_no = 1
;

### 壞人刺梅林
    UPDATE : GAME

In [ ]:
UPDATE GAME
SET assassin_target_seat_no = 3 -- 1 ~ player_count
WHERE game_id = '001'               -- 三位數 Game ID
;

### 刺完遊戲結束
    UPDATE : GAME

In [ ]:
UPDATE GAME
SET winner_side = 'Good'    -- Good or Evil
WHERE game_id = '001'         -- 三位數 Game ID
;

### 直接遊戲結束
    UPDATE : GAME

In [ ]:
UPDATE GAME
SET assassin_target_seat_no = NULL,
    winner_side = 'Evil'
WHERE game_id = '001'         -- 三位數 Game ID
;

### 公開角色牌（幾個人玩就要UPDATE幾次）
    UPDATE : PLAYER

In [ ]:
UPDATE PLAYER
SET role = '梅林'        -- 梅林、派西維爾、忠臣、刺客、魔甘娜、莫德雷德、奧伯倫、爪牙
WHERE game_id = '001'
AND seat_number = 1;

## 常用資料庫功能

### 清空數據

In [ ]:
TRUNCATE TABLE 表格名稱 CASCADE;

TRUNCATE TABLE 
    game, 
    player, 
    mission, 
    proposal, 
    proposal_contains, 
    vote_proposal, 
    becomes_mission 
RESTART IDENTITY CASCADE;


DROP TABLE IF EXISTS BECOMES_MISSION CASCADE;
DROP TABLE IF EXISTS VOTE_PROPOSAL CASCADE;
DROP TABLE IF EXISTS PROPOSAL_CONTAINS CASCADE;
DROP TABLE IF EXISTS PROPOSAL CASCADE;
DROP TABLE IF EXISTS MISSION CASCADE;
DROP TABLE IF EXISTS PLAYER CASCADE;
DROP TABLE IF EXISTS GAME CASCADE;

### 改Table名稱

In [ ]:
ALTER TABLE MISSION_INCLUDE RENAME TO MISSION_INCLUDES;

### 改欄位名稱

In [ ]:
ALTER TABLE MISSION_INCLUDE RENAME COLUMN include_or_not TO includes_or_not;

### 新增欄位

In [ ]:
ALTER TABLE PROPOSAL ADD COLUMN yes_amount INT;

### 刪除欄位

In [ ]:
ALTER TABLE VOTE_PROPOSAL DROP COLUMN yes_amount;